# Table of Contents
- [Description](#description)
    - [GSS approach](#gss-approach)
- [Gallup](#gallup)
    - [Approach: Embeddings](#approach-embeddings)
    - [Approach: Classifier](#approach-classifier)
    - [Hybrid](#hybrid)
    - [Extensions](#extensions)
    - [Observations](#observations)


# Description
This notebook is the workplace used to connect Gallup survey respones to an appropriate O\*NET occupation code.

# Gallup
For Gallup, it will use embeddings to determine the most appropriate job title (and associated O\*NET-SOC 2019 code) for the response.
- Some Gallup responses only have a categorical selection, whereas others have an additional text response the gives details on their occupation.
- Using the categorical selection **AND** the text response, an embedding will be generated.

## Approach: Embeddings
The closest O\*NET job title will be determined by fetching the job description that is the most similar to the Gallup survey response (with some similarity metric like `cosine similarity`, `L1`, or `L2 distance`).

## Approach: Classifier
The most fitting O\*NET job title will be determined by fetching the top 5 most similar O\*NET job titles (let it be referred to as `candidates`), passing the Gallup survey response and `candidates` to the classifier, and asking the classifier to choose the most fitting job title.

The survey response will then be assigned to the corresponding O\*NET-SOC 2019 occupation code.

## Hybrid
We can compare the two methods above and flag instances where the two disagree and ask for a human to determine which job title/occupation code is the most fitting for the survey response.

## Extensions
If time permits, an LLM call may be used. Similar to the classifier approach, the LLM will be given the `top k` closest job titles, and the Gallup response. It will be prompted to select the most appropriate job title for the Gallup response.

In [2]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

# Read in data
data_dir = "/content/drive/MyDrive/Swytch2A/Data/"
gallup_dir = data_dir + "gallup_survey/"
df_gallup = pd.read_csv(gallup_dir + "JFF_formatted_public_release_labels.csv",
                        encoding='cp1252')

Mounted at /content/drive


In [3]:
df_gallup.head()

,ENTITY_ID,SAMPLE,MODE,RESPONDENT_DATE,EndDate,S1,Q1,Q2,Q3,Q4,...,RACE6,RACE7,MARITAL_STATUS,EDUCATION,STATE,REGION,DIVISION,WORKER_TYPE,W2_STATUS,WEIGHT
0,4398199396,ABS Mail,Mail,1/28/2025,,,7,7,Very good,6,...,,White,Separated/divorced,High school graduate (high school diploma or e...,MA,Northeast,New England,Employee,W2 Worker,0.988427
1,4398199398,ABS Mail,Mail,1/28/2025,,,10-Completely satisfied,8,Excellent,10-Completely satisfied,...,,White,Married,"Four-year Bachelor's degree (e.g., BA, AB, BS)",MA,Northeast,New England,Employee,W2 Worker,0.184232
2,4398199412,ABS Mail,Mail,1/23/2025,,,8,8,Excellent,8,...,,White,Domestic partnership/Living with partner (not ...,"Advanced degree (Master's, professional, docto...",MA,Northeast,New England,Employee,W2 Worker,0.368465
3,4398199417,ABS Mail,Mail,1/30/2025,,,8,8,Very good,9,...,,White,Married,High school graduate (high school diploma or e...,MA,Northeast,New England,Self-employed IC,Non W2 Worker,3.341439
4,4398199431,ABS Web,Web,,1/16/2025 6:47:11,Yes,6,7,Very good,5,...,,,Domestic partnership/Living with partner (not ...,"Some college, no degree",MA,Northeast,New England,Employee,W2 Worker,1.515829


In [4]:
print(f"[INFO] Dataframe shape: {df_gallup.shape}")
print(f"[INFO] Dataframe columns: {list(df_gallup.columns)}")
print(f"[INFO] Dataframe number of job types: {df_gallup["Q6"].nunique()}")
print(f"[INFO] Dataframe number of industries: {df_gallup["Q7"].nunique()}")

[INFO] Dataframe shape: (18429, 153)
[INFO] Dataframe columns: ['ENTITY_ID', 'SAMPLE', 'MODE', 'RESPONDENT_DATE', 'EndDate', 'S1', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q6_CODED', 'Q6_TEXT', 'Q7', 'Q7_CODED', 'Q7_TEXT', 'Q8', 'Q9', 'Q10', 'Q11', 'Q12', 'Q12_TEXT', 'Q13', 'Q14', 'Q15', 'Q15_TEXT', 'Q16', 'Q16_TEXT', 'Q17', 'Q18', 'Q19', 'Q20', 'Q21', 'Q22', 'Q23', 'Q24', 'Q25A', 'Q25B', 'Q25C', 'Q25D', 'Q26_1', 'Q26_2', 'Q26_3', 'Q26_4', 'Q26_5', 'Q26_6', 'Q26_7', 'Q27', 'Q28', 'Q29', 'Q30', 'Q31A', 'Q31B', 'Q31C', 'Q32A', 'Q32B', 'Q32C', 'Q33A', 'Q33B', 'Q33C', 'Q33D', 'Q33E', 'Q33F', 'Q33G', 'Q33H', 'Q33I', 'Q33J', 'Q34', 'Q35A', 'Q35B', 'Q35C', 'Q35D', 'Q36', 'Q37A', 'Q37B', 'Q37C', 'Q37D', 'Q37E', 'Q37F', 'Q37G', 'Q37H', 'Q37I', 'Q38A', 'Q38B', 'Q38C', 'Q39', 'Q40', 'Q41', 'Q42', 'Q43', 'Q44', 'Q45', 'Q46', 'Q46_TEXT', 'Q47', 'Q48', 'Q49', 'Q49_TEXT', 'Q50', 'Q51', 'Q52', 'Q53', 'Q54A', 'Q54B', 'Q54C', 'Q55', 'Q55_TEXT', 'Q56_1', 'Q56_2', 'Q56_3', 'Q56_4', 'Q57', 'Q58', 'Q59', 'Q60',

In [5]:
print("[INFO] Types of categorical jobs:")
df_gallup["Q6"].unique()

[INFO] Types of categorical jobs:


array(['Building and grounds cleaning and maintenance',
       'Business and financial operations', 'Computer and mathematical',
       'Construction and extraction', 'Office and administrative support',
       'Installation, maintenance, and repair', 'Healthcare technicians',
       'Healthcare practitioners', 'Educational instruction and library',
       'Protective service', 'Something else (please specify):',
       'Management', 'Life, physical, and social science',
       'Sales and related', 'Healthcare support', 'Production',
       'Transportation and material moving',
       'Architecture and engineering', 'Food preparation and service',
       'Legal', 'Arts, design, entertainment, sports, and media',
       'Community and social service', 'Personal care and service',
       'Military', 'No response', 'Farming, fishing, and forestry'],
      dtype=object)

In [6]:
print("[INFO] Types of categorical industries:")
df_gallup["Q7"].unique()

[INFO] Types of categorical industries:


array(['Health/medical', 'Education', 'Construction', 'Government',
       'Farming, agriculture, and mining/extraction (e.g., coal, oil, gas)',
       'Professional and business services', 'Retail trade',
       'Other services', 'Leisure and hospitality', 'Social services',
       'Manufacturing', 'Wholesale trade', 'Transportation',
       'Financial activities',
       'You work for a private individual or household', 'Utilities',
       'Information (e.g., publishing, news media, motion pictures)',
       'Warehousing', 'Something else (please specify):', 'No response'],
      dtype=object)

In [7]:
# Remove nullish values from DataFrame and replace with nan
df_gallup = df_gallup.replace(r"^\s*$|No response|-98", np.nan, regex=True)
df_gallup = df_gallup.replace(-98, np.nan, regex=True)

df_gallup.head()

,ENTITY_ID,SAMPLE,MODE,RESPONDENT_DATE,EndDate,S1,Q1,Q2,Q3,Q4,...,RACE6,RACE7,MARITAL_STATUS,EDUCATION,STATE,REGION,DIVISION,WORKER_TYPE,W2_STATUS,WEIGHT
0,4398199396,ABS Mail,Mail,1/28/2025,NaN,NaN,7,7,Very good,6,...,NaN,White,Separated/divorced,High school graduate (high school diploma or e...,MA,Northeast,New England,Employee,W2 Worker,0.988427
1,4398199398,ABS Mail,Mail,1/28/2025,NaN,NaN,10-Completely satisfied,8,Excellent,10-Completely satisfied,...,NaN,White,Married,"Four-year Bachelor's degree (e.g., BA, AB, BS)",MA,Northeast,New England,Employee,W2 Worker,0.184232
2,4398199412,ABS Mail,Mail,1/23/2025,NaN,NaN,8,8,Excellent,8,...,NaN,White,Domestic partnership/Living with partner (not ...,"Advanced degree (Master's, professional, docto...",MA,Northeast,New England,Employee,W2 Worker,0.368465
3,4398199417,ABS Mail,Mail,1/30/2025,NaN,NaN,8,8,Very good,9,...,NaN,White,Married,High school graduate (high school diploma or e...,MA,Northeast,New England,Self-employed IC,Non W2 Worker,3.341439
4,4398199431,ABS Web,Web,NaN,1/16/2025 6:47:11,Yes,6,7,Very good,5,...,NaN,NaN,Domestic partnership/Living with partner (not ...,"Some college, no degree",MA,Northeast,New England,Employee,W2 Worker,1.515829


In [8]:
print(f"[INFO] Number of text responses for Q6: {df_gallup["Q6_TEXT"].notna().sum()}")
print(f"[INFO] Number of text responses for Q7: {df_gallup["Q7_TEXT"].notna().sum()}")

[INFO] Number of text responses for Q6: 2447
[INFO] Number of text responses for Q7: 2117


In [9]:
print(f"[CHECK] Q6 category null but Q6 text not null: {(df_gallup["Q6"].isna() & df_gallup["Q6_TEXT"].notna()).sum()}")
print(f"[CHECK] Q7 category null but Q7 text not null: {(df_gallup["Q7"].isna() & df_gallup["Q7_TEXT"].notna()).sum()}")

[CHECK] Q6 category null but Q6 text not null: 0
[CHECK] Q7 category null but Q7 text not null: 0


## Observations
From the Gallup codebook, it looks like -98 and "No response" are values for nullish responses. Therefore, they were replaced with `np.nan`.

In addition, it looks like we only have text inputs for job info and industry info if the response has also selected a job category and industry category, respectively.

## Embedding O\*NET job descriptions and associating with job codes

In [10]:
# Load occupation data
df_onet = pd.read_csv(data_dir + "onet_misc/occupation_data.csv")
print(f"[INFO] Number of O*NET jobs: {df_onet.shape[0]}")
print(f"[INFO] O*NET job info columns: {list(df_onet.columns)}")
df_onet.head()

[INFO] Number of O*NET jobs: 1016
[INFO] O*NET job info columns: ['O*NET-SOC Code', 'Title', 'Description']


,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."


In [11]:
# Create list of formatted job titles to job descriptions to embed
onet_jd_title = list(f"{title}: {jd}" for title, jd
                     in zip(df_onet["Title"], df_onet["Description"]))

# Map for job title to code
title_to_code = { title:code for code, title
                 in list(zip(df_onet["O*NET-SOC Code"], df_onet["Title"])) }

In [12]:
import torch
from sentence_transformers import SentenceTransformer

# Embed our formatted job titles to job descriptions
model = SentenceTransformer("all-MiniLM-L6-v2")
device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings = model.encode(
    onet_jd_title,
    device=device,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [13]:
def get_k_most_similar(gallup_phrase, onet_embeddings, df_onet, k=1):
    embedding = model.encode(gallup_phrase, normalize_embeddings=True)
    similarities = onet_embeddings @ embedding
    top_k_idx = np.argsort(similarities)[-k:][::-1]
    return [(df_onet.iloc[i]["O*NET-SOC Code"], df_onet.iloc[i]["Title"]) for i in top_k_idx]

In [14]:
def get_onet_code_embed_only(phrase, onet_embeddings, df_onet):
  code_title = get_k_most_similar(phrase, onet_embeddings, df_onet)
  if code_title is None:
      return np.nan
  return code_title[0][0]

def get_onet_code_clf(clf, job_desc, candidates, title_to_code):
  template = "This job description matches the occupation: {}"
  result = clf(job_desc, candidate_labels=candidates, hypothesis_template=template)
  return title_to_code.get(result["labels"][0], None)

In [15]:
# Test template with a response pulled from the dataset
phrase = '''At my job, I work in/as arts, design, entertainment, sports, and media.
More specifically, I work in/as technical designer for uniform clothing company.
My job is related to the manufacturing industry.
More specifically, my job is related to the uniform clothing manufacturer industry.'''
print(f"[TEST] Phrase: \"{phrase}\"\nCode: {get_onet_code_embed_only(phrase,
                                                                     embeddings,
                                                                     df_onet)}")
test_k = get_k_most_similar(phrase, embeddings, df_onet, k=5)
print(test_k)

[TEST] Phrase: "At my job, I work in/as arts, design, entertainment, sports, and media.
More specifically, I work in/as technical designer for uniform clothing company.
My job is related to the manufacturing industry.
More specifically, my job is related to the uniform clothing manufacturer industry."
Code: 27-1022.00
[('27-1022.00', 'Fashion Designers'), ('15-1299.00', 'Computer Occupations, All Other'), ('41-4012.00', 'Sales Representatives, Wholesale and Manufacturing, Except Technical and Scientific Products'), ('51-6052.00', 'Tailors, Dressmakers, and Custom Sewers'), ('25-2032.00', 'Career/Technical Education Teachers, Secondary School')]


In [16]:
from transformers import pipeline

# Classifier model
classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33",
    device=0,
)

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  142MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [17]:
# Test classifier approach with the same phrase
result = get_onet_code_clf(classifier,
                           phrase,
                           [title for _, title in test_k],
                           title_to_code)
print(f"[TEST CLF] Phrase: \"{phrase}\"\nCode: {result}")

[TEST CLF] Phrase: "At my job, I work in/as arts, design, entertainment, sports, and media.
More specifically, I work in/as technical designer for uniform clothing company.
My job is related to the manufacturing industry.
More specifically, my job is related to the uniform clothing manufacturer industry."
Code: 27-1022.00


In [18]:
def build_templated_str(row):
    Q6_categorical_template = "At my job, I work in/as {}."
    Q6_text_template = "More specifically, I work in/as {}."
    Q7_categorical_template = "My job is related to the {} industry."
    Q7_text_template = "More specifically, my job industry is in {}"

    parts = []
    if pd.notna(row["Q6"]):
        normalized = row["Q6"].strip().lower()
        parts.append(Q6_categorical_template.format(normalized))
    if pd.notna(row["Q6_TEXT"]):
        normalized = row["Q6_TEXT"].strip().lower()
        parts.append(Q6_text_template.format(normalized))
    if pd.notna(row["Q7"]):
        normalized = row["Q7"].strip().lower()
        parts.append(Q7_categorical_template.format(normalized))
    if pd.notna(row["Q7_TEXT"]):
        normalized = row["Q7_TEXT"].strip().lower()
        parts.append(Q7_text_template.format(normalized))

    return ' '.join(parts)

In [19]:
# For each response, build the templated job description for the survey response
mask = (df_gallup["Q6"].notna() | df_gallup["Q7"].notna())
df_gallup_response_rows = df_gallup[mask]
df_gallup_response_rows["templated_str"] = df_gallup_response_rows.apply(
    build_templated_str,
    axis=1)

In [20]:
def embed_write(df_gallup, df_gallup_response_rows):
  # Iterate and determine which job title fits the best with the current response
  curr_iter = 0
  for idx, row in df_gallup_response_rows.iterrows():
      job_text = row["templated_str"]
      top_k = get_k_most_similar(job_text, embeddings, df_onet, k=1)
      if top_k is None or len(top_k) == 0:
        print(f"[WARNING] No fit found for iter {curr_iter}")

      code = top_k[0][0]
      df_gallup.loc[idx, "ONET_SOC_CODE"] = code

      curr_iter += 1
      if curr_iter % 100 == 0:
          print(f"[INFO] Finished iteration {curr_iter}.")
          df_gallup.to_csv(gallup_dir + "gallup_with_onet_embed.csv", index=False)

In [21]:
def clf_write(df_gallup, df_gallup_response_rows, title_to_code, clf):
  # Iterate and determine which job title fits the best with the current response
  curr_iter = 0
  for idx, row in df_gallup_response_rows.iterrows():
      job_text = row["templated_str"]
      top_k = get_k_most_similar(job_text,
                                embeddings,
                                df_onet,
                                k=5)
      top_k = [title for _, title in top_k]
      result = get_onet_code_clf(classifier, job_text, top_k, title_to_code)
      df_gallup.loc[idx, "ONET_SOC_CODE"] = result

      curr_iter += 1
      if curr_iter % 100 == 0:
          print(f"[INFO] Finished iteration {curr_iter}.")
          df_gallup.to_csv(gallup_dir + "gallup_with_onet_clf.csv", index=False)

In [22]:
def check_null(df_gallup):
  # Check that all responses have an associated O*NET-SOC Occupation code
  has_null = ("None" in df_gallup["ONET_SOC_CODE"].unique()
              or None in df_gallup["ONET_SOC_CODE"].unique())

  if has_null:
      print("[WARNING] There exist rows that have not been populated with an O*NET code.")
  else:
      print("[INFO] All rows have an associated O*NET code.")

def reorder_cols(df_gallup):
  # Reorder columns so that O*NET-SOC code is towards the front of the dataset
  prev_shape = df_gallup.shape
  prev_cols = set(df_gallup.columns)

  prefix_cols = ["ENTITY_ID", "SAMPLE", "MODE", "RESPONDENT_DATE", "EndDate", "ONET_SOC_CODE"]
  cols = prefix_cols + [col for col in df_gallup.columns if col not in prefix_cols]
  df_gallup = df_gallup[cols]

  curr_shape = df_gallup.shape
  curr_cols = set(df_gallup.columns)
  same_cols = len(curr_cols - prev_cols) == 0 and prev_shape == curr_shape
  if not same_cols:
      print("[WARNING] Columns before and after reordering are not the same.")
  else:
      print("[INFO] Columns before and after reordering are the same")
  print(f"[INFO] Shape before: {prev_shape}, shape after: {curr_shape}")

  return df_gallup

### Executing Embedded Approach

In [23]:
embed_write(df_gallup, df_gallup_response_rows)

[INFO] Finished iteration 100.
[INFO] Finished iteration 200.
[INFO] Finished iteration 300.
[INFO] Finished iteration 400.
[INFO] Finished iteration 500.
[INFO] Finished iteration 600.
[INFO] Finished iteration 700.
[INFO] Finished iteration 800.
[INFO] Finished iteration 900.
[INFO] Finished iteration 1000.
[INFO] Finished iteration 1100.
[INFO] Finished iteration 1200.
[INFO] Finished iteration 1300.
[INFO] Finished iteration 1400.
[INFO] Finished iteration 1500.
[INFO] Finished iteration 1600.
[INFO] Finished iteration 1700.
[INFO] Finished iteration 1800.
[INFO] Finished iteration 1900.
[INFO] Finished iteration 2000.
[INFO] Finished iteration 2100.
[INFO] Finished iteration 2200.
[INFO] Finished iteration 2300.
[INFO] Finished iteration 2400.
[INFO] Finished iteration 2500.
[INFO] Finished iteration 2600.
[INFO] Finished iteration 2700.
[INFO] Finished iteration 2800.
[INFO] Finished iteration 2900.
[INFO] Finished iteration 3000.
[INFO] Finished iteration 3100.
[INFO] Finished i

In [24]:
check_null(df_gallup)
df_gallup = reorder_cols(df_gallup)

[INFO] All rows have an associated O*NET code.
[INFO] Columns before and after reordering are the same
[INFO] Shape before: (18429, 154), shape after: (18429, 154)


In [25]:
# Write to CSV file
df_gallup.to_csv(gallup_dir + "gallup_with_onet_embed.csv", index=False)

### Executing classifier approach

In [26]:
clf_write(df_gallup, df_gallup_response_rows, title_to_code, classifier)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[INFO] Finished iteration 100.
[INFO] Finished iteration 200.
[INFO] Finished iteration 300.
[INFO] Finished iteration 400.
[INFO] Finished iteration 500.
[INFO] Finished iteration 600.
[INFO] Finished iteration 700.
[INFO] Finished iteration 800.
[INFO] Finished iteration 900.
[INFO] Finished iteration 1000.
[INFO] Finished iteration 1100.
[INFO] Finished iteration 1200.
[INFO] Finished iteration 1300.
[INFO] Finished iteration 1400.
[INFO] Finished iteration 1500.
[INFO] Finished iteration 1600.
[INFO] Finished iteration 1700.
[INFO] Finished iteration 1800.
[INFO] Finished iteration 1900.
[INFO] Finished iteration 2000.
[INFO] Finished iteration 2100.
[INFO] Finished iteration 2200.
[INFO] Finished iteration 2300.
[INFO] Finished iteration 2400.
[INFO] Finished iteration 2500.
[INFO] Finished iteration 2600.
[INFO] Finished iteration 2700.
[INFO] Finished iteration 2800.
[INFO] Finished iteration 2900.
[INFO] Finished iteration 3000.
[INFO] Finished iteration 3100.
[INFO] Finished i

In [27]:
check_null(df_gallup)
df_gallup = reorder_cols(df_gallup)

[INFO] All rows have an associated O*NET code.
[INFO] Columns before and after reordering are the same
[INFO] Shape before: (18429, 154), shape after: (18429, 154)


In [28]:
# Write to CSV file
df_gallup.to_csv(gallup_dir + "gallup_with_onet_clf.csv", index=False)